# PyTorch Profiler Example

Demonstrates how to use `torch.profiler` to measure the runtime of CUDA tensor operations.


In [1]:
import torch
import torch.profiler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Example: matrix multiply + activation on a large tensor
def workload(x):
    w1 = torch.randn(2048, 2048, device=device, dtype=torch.float64)
    w2 = torch.randn(2048, 2048, device=device, dtype=torch.float64)
    out = torch.mm(x, w1)          # matrix multiply
    out = torch.relu(out)           # activation
    out = torch.mm(out, w2)        # second matmul
    out = torch.softmax(out, dim=1) # softmax
    return out

x = torch.randn(512, 2048, device=device, dtype=torch.float64)

# Warm-up (important for CUDA — avoids one-time JIT costs in profile)
with torch.no_grad():
    workload(x)

# Profile the workload
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    record_shapes=True,         # record tensor shapes
    profile_memory=True,        # track memory allocation
    with_stack=False,
) as prof:
    with torch.no_grad():
        for _ in range(5):      # run a few iterations for stable measurements
            workload(x)

# ── Print top operators sorted by CUDA time ──────────────────────────────────
print("\n=== Top 15 ops by CUDA time ===")
print(
    prof.key_averages().table(
        sort_by="cuda_time_total",
        row_limit=15,
    )
)

# ── Print top operators sorted by CPU time ───────────────────────────────────
print("\n=== Top 15 ops by CPU time ===")
print(
    prof.key_averages().table(
        sort_by="cpu_time_total",
        row_limit=15,
    )
)


Using device: cuda

=== Top 15 ops by CUDA time ===
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         0.14%     102.000us         0.20%     149.000us      14.900us      64.410ms        91.88%      64.678ms       6.468ms   

STAGE:2026-04-03 07:13:38 629066:629066 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
STAGE:2026-04-03 07:13:38 629066:629066 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2026-04-03 07:13:38 629066:629066 ActivityProfilerController.cpp:324] Completed Stage: Post Processing


In [2]:
# ── Export a Chrome-trace JSON for the Perfetto / chrome://tracing viewer ──
# Uncomment to save:
prof.export_chrome_trace("trace.json")

# ── Per-operator breakdown with input shapes ─────────────────────────────────
print("=== Average stats per op (with input shapes) ===")
for evt in prof.key_averages(group_by_input_shape=True):
    if evt.cuda_time_total > 0:
        print(
            f"{evt.key:40s}  "
            f"CPU: {evt.cpu_time_total/1e3:8.3f} ms  "
            f"CUDA: {evt.cuda_time_total/1e3:8.3f} ms  "
            f"input shapes: {evt.input_shapes}"
        )


=== Average stats per op (with input shapes) ===
aten::randn                               CPU:    0.954 ms  CUDA:    5.330 ms  input shapes: [[], [], [], [], []]
aten::normal_                             CPU:    0.099 ms  CUDA:    5.330 ms  input shapes: [[2048, 2048], [], [], []]
cudaStreamIsCapturing                     CPU:    0.003 ms  CUDA:    0.268 ms  input shapes: []
aten::mm                                  CPU:    0.149 ms  CUDA:   64.678 ms  input shapes: [[512, 2048], [2048, 2048]]
cudaMemsetAsync                           CPU:    0.021 ms  CUDA:    0.268 ms  input shapes: []
aten::relu                                CPU:    0.054 ms  CUDA:    0.046 ms  input shapes: [[512, 2048]]
aten::clamp_min                           CPU:    0.039 ms  CUDA:    0.046 ms  input shapes: [[512, 2048], []]
aten::softmax                             CPU:    0.040 ms  CUDA:    0.465 ms  input shapes: [[512, 2048], [], []]
aten::_softmax                            CPU:    0.031 ms  CUDA:    0.

In [3]:
import torch
import torch.profiler
from contextlib import contextmanager

@contextmanager
def profile_block(label="profiled block", sort_by="cuda_time_total", row_limit=15, export_trace=None):
    """
    Context manager that profiles any code block placed inside it.

    Usage:
        with profile_block("my op"):
            # put any code here (no JIT restriction)
            result = some_tensor_op(x)

    Args:
        label       : descriptive name printed in the header
        sort_by     : "cuda_time_total" | "cpu_time_total" | "self_cuda_memory_usage"
        row_limit   : how many rows to show in the summary table
        export_trace: optional file path to write a Chrome-trace JSON
    """
    with torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU,
            torch.profiler.ProfilerActivity.CUDA,
        ],
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        yield prof   # your code runs here

    print(f"\n{'='*60}")
    print(f" Profile: {label}  (sorted by {sort_by})")
    print('='*60)
    print(prof.key_averages().table(sort_by=sort_by, row_limit=row_limit))

    if export_trace:
        prof.export_chrome_trace(export_trace)
        print(f"Chrome trace saved to: {export_trace}")


# ── Example: profile any arbitrary block ─────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N = 4096

a = torch.randn(N, N, device=device)
b = torch.randn(N, N, device=device)

# Warm up outside the profiler so one-time CUDA costs don't pollute results
with torch.no_grad():
    _ = torch.mm(a, b)

# Drop any code block here to profile it
with profile_block("matmul + FFT + conv1d", sort_by="cuda_time_total"):
    with torch.no_grad():
        # matrix multiply
        c = torch.mm(a, b)
        # element-wise FFT
        f = torch.fft.fft(c)
        # 1-D convolution  (batch=1, channels=N, length=N)
        x1d = a.unsqueeze(0)                           # (1, N, N)
        kernel = torch.randn(32, N, 3, device=device)  # 32 output channels
        out = torch.nn.functional.conv1d(x1d, kernel, padding=1)


STAGE:2026-04-03 07:23:00 629066:629066 ActivityProfilerController.cpp:314] Completed Stage: Warm Up



 Profile: matmul + FFT + conv1d  (sorted by cuda_time_total)
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         0.32%     962.000us         0.35%       1.076ms       1.076ms       2.430ms        52.20%       2.430ms       

STAGE:2026-04-03 07:23:00 629066:629066 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2026-04-03 07:23:00 629066:629066 ActivityProfilerController.cpp:324] Completed Stage: Post Processing


In [ ]:
import subprocess
import sys
import os
import torch
import torch.profiler
import tempfile
import json
from pathlib import Path


def profile_script(
    script_path: str,
    *script_args: str,
    warmup_runs: int = 0,
    sort_by: str = "cuda_time_total",
    row_limit: int = 20,
    export_trace: str | None = None,
    python_executable: str = sys.executable,
) -> None:
    """
    Profile the runtime of an entire Python script by injecting
    torch.profiler instrumentation around its execution.

    The target script is run inside the same Python process via
    `runpy.run_path`, so all torch CUDA context is shared and no
    subprocess overhead is added to the measurements.

    Args:
        script_path       : path to the .py script to profile
        *script_args      : command-line arguments forwarded to sys.argv
                            inside the script (sys.argv[0] = script_path)
        warmup_runs       : number of un-profiled warm-up executions
                            before the profiled run (default 0)
        sort_by           : profiler sort key —
                            "cuda_time_total" | "cpu_time_total" |
                            "self_cuda_memory_usage" | "self_cpu_memory_usage"
        row_limit         : rows shown in the summary table
        export_trace      : optional path for a Chrome-trace JSON file
        python_executable : Python used for validation (default: current)
    """
    import runpy

    script_path = str(Path(script_path).resolve())
    if not os.path.isfile(script_path):
        raise FileNotFoundError(f"Script not found: {script_path}")

    # Patch sys.argv so argparse / sys.argv usage inside the script works
    original_argv = sys.argv[:]
    sys.argv = [script_path, *script_args]

    try:
        # ── Optional warm-up runs (un-profiled) ──────────────────────────────
        for i in range(warmup_runs):
            print(f"[profile_script] warm-up run {i + 1}/{warmup_runs} ...")
            runpy.run_path(script_path, run_name="__main__")

        # ── Profiled run ─────────────────────────────────────────────────────
        print(f"[profile_script] profiling: {script_path}")
        with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA,
            ],
            record_shapes=True,
            profile_memory=True,
            with_stack=False,
        ) as prof:
            runpy.run_path(script_path, run_name="__main__")

    finally:
        sys.argv = original_argv  # always restore sys.argv

    # ── Results ──────────────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f" Profile of: {os.path.basename(script_path)}")
    print(f" Sorted by : {sort_by}")
    print(f"{'='*65}")
    print(prof.key_averages().table(sort_by=sort_by, row_limit=row_limit))

    if export_trace:
        prof.export_chrome_trace(export_trace)
        print(f"[profile_script] Chrome trace saved -> {export_trace}")

    return prof


# ── Demo: write a tiny throw-away script and profile it ──────────────────────

_demo_script = """\
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N = 2048

a = torch.randn(N, N, device=device)
b = torch.randn(N, N, device=device)

# some typical operations
c = torch.mm(a, b)
d = torch.nn.functional.layer_norm(c, c.shape[-1:])
e = torch.fft.rfft2(d)
"""

# Write the demo script to a temp file
_tmp = tempfile.NamedTemporaryFile(suffix=".py", mode="w", delete=False)
_tmp.write(_demo_script)
_tmp.flush()
_tmp_path = _tmp.name

try:
    prof_result = profile_script(
        _tmp_path,
        sort_by="cuda_time_total",
        row_limit=15,
        export_trace="demo_trace.json",   # remove / comment out if not needed
    )
finally:
    os.unlink(_tmp_path)
